# Agilent LCR & Janis Temperature Controller
Automated Measurements with Live Plotting

In [ ]:
# Imports
from nfoinstruments.drivers import Janis, E4890A
from nfoinstruments.drivers.setup import MeasurementSetup
from nfoinstruments.procedures import (
    set_temperature_and_wait, 
    sweep_frequency_lcr,
    single_frequency_time_scan,
    set_bias_and_wait,
    build_cv_bias_path,
    sweep_cv_lcr,
    load_measurement_files,
    load_cv_measurement_files,
    plot_all_measurements,
    plot_measurement_comparison,
    plot_time_scan_comparison,
    plot_cv_comparison
)
from nfoinstruments.procedures.utils import run_temperature_bias_sweep_with_live_plot, run_cv_sweep_with_live_plot

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from pathlib import Path

# Output configuration
output_path = Path(r"C:\Users\F110216\Desktop\Data_Horatio\NewExperiment")

In [ ]:
# Initialize instruments
mm = MeasurementSetup()

# Connect to devices (update GPIB addresses if needed)
mm.connect_to_devices({
    'GPIB1::16::INSTR': Janis,
    'GPIB1::17::INSTR': E4890A
})

janis = mm.devices['GPIB1::16::INSTR']
lcr = mm.devices['GPIB1::17::INSTR']

# Configure LCR meter
lcr.measurement_type = E4890A.MeasurementType.ZTD  # Options: ZTD (Z, θ), RX (R, X), CPD (Cp, D), CSD (Cs, D)...
lcr.measurement_time = E4890A.MeasurementTime.LONG # Options: SHORT, MEDIUM, LONG
lcr.averages = 1                                 # 1 to 256
lcr.signal_type = E4890A.SignalType.VOLTAGE        # VOLTAGE or CURRENT
lcr.signal_amplitude = 0.05                       # Signal Amplitude (V or A)
lcr.bias = 0.0                                     # DC Bias (V)
lcr.alc_enabled = True                             # Automatic Level Control (True/False)

print(f"LCR configured:")
print(f"  Type: {lcr.measurement_type.name}")
print(f"  Signal: {lcr.signal_amplitude} ({lcr.signal_type.name})")
print(f"  Time: {lcr.measurement_time.name} (Avg: {lcr.averages})")
print(f"  ALC: {lcr.alc_enabled}")

## 1. Impedance Spectroscopy (IS) Scans

In [ ]:
# Define parameters for IS Sweep
# 60 points per decade, from 20 Hz to 2 MHz (5 decades -> 300 points)
frequencies = np.round(np.logspace(np.log10(20), np.log10(2e6), num=200), decimals=2)

temperatures = list(range(300, 0, -20)) #(start,stop,step) go down to stop-step K in steps of step K

dc_biases = [0.0, -2.0, -1.0, 0.0, 1.0, 2.0, 0.0, -2.0, -1.0, 0.0, 1.0, 2.0, 0.0]  # V

sweep_name = "Run1_IS"

# Run measurement
next_run = run_temperature_bias_sweep_with_live_plot(
    parent_dir=output_path,
    sweep_name=sweep_name,
    temp_points=temperatures,
    bias_points=dc_biases,
    janis_ctrl=janis,
    lcr_ctrl=lcr,
    freq_points=frequencies,
    run_count_start=1
)

## 2. Capacitance-Voltage (CV) Scans

In [ ]:
# Define parameters for CV Sweep
# CV loop shape: 0 -> +Vmax -> Vmin -> 0
Vmin = -5.0
Vmax = 5.0
Vstep = 0.5

temperatures = [300, 250, 200, 150]  # K

sweep_name = "Run2_CV"

# AC condition 
Vrms = 0.1

# Frequency list for CV scans
cv_freq_points = [1e4, 1e5]


# Set up cycling to repeat the measurement with the same conditions 
cycles = 1 # Number of times to repeat the entire CV sequence at each temperature

# Run measurement
next_run = run_cv_sweep_with_live_plot(
    parent_dir=output_path,
    sweep_name="CV_" + sweep_name,
    temp_points=temperatures,
    freq_points=cv_freq_points,
    Vmin=Vmin,
    Vmax=Vmax,
    Vstep=Vstep,
    Vrms=Vrms,
    cycles=cycles,
    janis_ctrl=janis,
    lcr_ctrl=lcr,
    run_count_start=next_run
)